In [1]:
import os
from pathlib import Path
from ures.files import filter_files
from perf_estimator.estimator import Estimator, TrainerEstimator
from perf_estimator.dataset import image_dataset
from experiments.snapshot import SnapshotAnalyser
from perf_estimator.profiler import ProfilerDataProcessing
from ures.string import format_memory

from perf_estimator.utilis import format_memory


In [7]:
# Setup Basic Global Variables
# root_dir = Path("/Users/jiaboshi/Documents/101-Data/002-xMem-LLM")
root_dir = Path("/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/")
pytorch_dir = root_dir / "001-PyTorch"
huggingface_dir = root_dir / "002-HuggingFace"

pytorch_dir_name_format = "recurrence-{}-SGD-{}-1"
huggingface_dir_name_format_xmem = "{}-{}-xMem"
huggingface_dir_name_format_llm = "{}-{}-LLM"
huggingface_dir_name_format_cuda = "{}-{}-CUDA"

In [3]:
model = "VGG16"
batch_size = "170"
max_gpu_memory = 8

In [4]:
def get_xmem_memory(profiler_file, batch, max_in_gb, huggingface_enabled=False):
    if huggingface_enabled:
        estimator = TrainerEstimator(
            dataloader=image_dataset(batch=int(batch)),
            profiler_file=profiler_file,
            max_gpu_memory_in_gb=max_in_gb
        )
    else:
        estimator = Estimator(
            dataloader=image_dataset(batch=int(batch)),
            profiler_file=profiler_file,
            max_gpu_memory_in_gb=max_in_gb
        )
    my_result, _ = estimator.estimate()
    return max(my_result._trace.max_segment_changes), estimator


def get_ground_value_from_snapshot(snapshot_file: str) -> dict:
    _snapshot = SnapshotAnalyser(snapshot_file)
    return _snapshot.gpu_and_segment_in_same_time_length()

import re

def extract_bytes(mem_list):
    """
    从每个内存条目中提取 Bytes 数值，返回排序后的数值列表。
    假设每个内存条目是一个字符串，格式中包含 "Bytes:<数字>"
    """
    bytes_values = []
    pattern = re.compile(r'Bytes:(\d+)')
    for item in mem_list:
        # 将 item 转换为字符串以防万一
        text = str(item)
        match = pattern.search(text)
        if match:
            bytes_values.append(int(match.group(1)))
    return sorted(bytes_values)

def compare_memory_bytes(list1, list2, debug=False):
    """
    比较两个列表中各模块的 forward_memory 和 backward_memory 中的 bytes 数值是否一致
    """
    # 构造成字典，以模块名称为 key
    dict1 = {entry['name']: entry for entry in list1}
    dict2 = {entry['name']: entry for entry in list2}
    forward_matched = True
    backwrd_matched = True

    # 得到所有模块名称
    all_names = set(dict1.keys()).union(dict2.keys())

    for name in all_names:
        entry1 = dict1.get(name)
        entry2 = dict2.get(name)

        if entry1 is None:
            print(f"模块 {name} 仅存在于第二个列表中。")
            continue
        if entry2 is None:
            print(f"模块 {name} 仅存在于第一个列表中。")
            continue

        # 分别提取 forward_memory 和 backward_memory 中的 bytes 数值
        forward_bytes1 = extract_bytes(entry1.get('forward_memory', []))
        forward_bytes2 = extract_bytes(entry2.get('forward_memory', []))
        backward_bytes1 = extract_bytes(entry1.get('backward_memory', []))
        backward_bytes2 = extract_bytes(entry2.get('backward_memory', []))

        if forward_bytes1 == forward_bytes2:
            if debug:
                print(f"{name}: forward_memory 的 bytes 匹配")
        else:
            forward_matched = False
            if debug:
                print(f"{name}: forward_memory 的 bytes 不匹配")
                print(f"  List1: {forward_bytes1}")
                print(f"  List2: {forward_bytes2}")

        if backward_bytes1 == backward_bytes2:
            if debug:
                print(f"{name}: backward_memory 的 bytes 匹配")
        else:
            backwrd_matched = False
            if debug:
                print(f"{name}: backward_memory 的 bytes 不匹配")
                print(f"  List1: {backward_bytes1}")
                print(f"  List2: {backward_bytes2}")

    return forward_matched, backwrd_matched


In [6]:
from typing import Union
def get_all_memory_information(model_name: str, batch: Union[str, int]) -> dict:
    batch = str(batch)
    # Get PyTorch Data
    torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
    all_torch_dirs = os.listdir(torch_data_dir)
    all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False]
    dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)
    torch_snapshot_file = filter_files(".pickle", dirs_sorted[0], fuzz=True)[-1]
    torch_profiler_file = filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1]

    # Get HuggingFace Data
    huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
    huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
    huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
    huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
    huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
    huggingface_snapshot_file_xmem = filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1]

    # Estimate memory
    huggingface_memory_llm, huggingface_estimator = get_xmem_memory(
        profiler_file=huggingface_profiler_file_llm,
        batch=batch_size,
        max_in_gb=max_gpu_memory,
        huggingface_enabled=True
    )
    huggingface_snapshot_memory_xmen = max(get_ground_value_from_snapshot(huggingface_snapshot_file_xmem)['seg'])
    huggingface_memory_diff = huggingface_memory_llm - huggingface_snapshot_memory_xmen

    paper_memory_result, paper_estimator = get_xmem_memory(
        profiler_file=torch_profiler_file,
        batch=batch_size,
        max_in_gb=max_gpu_memory
    )
    paper_snapshot_result = max(get_ground_value_from_snapshot(torch_snapshot_file)['seg'])
    paper_memory_diff = paper_memory_result - paper_snapshot_result

    forward_matched, backward_matched = compare_memory_bytes(
        paper_estimator.profiler.get_iteration(1).layer_summary(),
        huggingface_estimator.profiler.get_iteration(1).layer_summary()
    )

    return {
        "torch": {
            "est": paper_memory_result,
            "ground": paper_snapshot_result,
            "error": round((abs(paper_memory_result - paper_snapshot_result)/paper_snapshot_result)*100, 2),
            "diff": paper_memory_diff
        },
        "huggingface": {
            "est": huggingface_memory_llm,
            "ground": huggingface_snapshot_memory_xmen,
            "error": round((abs(huggingface_memory_llm - huggingface_snapshot_memory_xmen)/huggingface_snapshot_memory_xmen)*100, 2),
            "diff": huggingface_memory_diff
        },
        "compare": {
            "forward": forward_matched,
            "backward": backward_matched,
            "est_diff": huggingface_memory_llm - paper_memory_result,
        }
    }


In [7]:
models = ["ConvNeXtTiny", "ResNet50", "VGG16"]
batch = range(10, 570, 40)

In [8]:
result_list = []
for m in models:
    for b in batch:
        _r = get_all_memory_information(m, b)
        _r.update({
            "model": m,
            "batch": b
        })
        result_list.append(_r)

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_92
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_3c
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_3b
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_49
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_30
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_2b
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_53
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_37
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_78
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_65
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_58
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_d4
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_22
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_47
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_71
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_df
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_43
Duplicate laye

模块 ReLU_4_37 仅存在于第二个列表中。
模块 ReLU_14_fd 仅存在于第一个列表中。
模块 ReLU_5_6c 仅存在于第一个列表中。
模块 ReLU_10_f7 仅存在于第二个列表中。
模块 ReLU_13_ef 仅存在于第一个列表中。
模块 ReLU_15_4f 仅存在于第二个列表中。
模块 ReLU_14_02 仅存在于第二个列表中。
模块 ReLU_7_47 仅存在于第二个列表中。
模块 ReLU_2_49 仅存在于第二个列表中。
模块 ReLU_6_58 仅存在于第二个列表中。
模块 ReLU_1_a9 仅存在于第一个列表中。
模块 ReLU_16_05 仅存在于第二个列表中。
模块 ReLU_2_d4 仅存在于第一个列表中。
模块 ReLU_7_22 仅存在于第二个列表中。
模块 ReLU_15_7f 仅存在于第二个列表中。
模块 ReLU_8_df 仅存在于第二个列表中。
模块 ReLU_3_c8 仅存在于第一个列表中。
模块 ReLU_8_10 仅存在于第一个列表中。
模块 ReLU_5_65 仅存在于第二个列表中。
模块 ReLU_15_db 仅存在于第一个列表中。
模块 ReLU_5_78 仅存在于第二个列表中。
模块 ReLU_8_71 仅存在于第二个列表中。
模块 ReLU_1_87 仅存在于第一个列表中。
模块 ReLU_7_58 仅存在于第一个列表中。
模块 ReLU_13_e4 仅存在于第一个列表中。
模块 ReLU_14_37 仅存在于第二个列表中。
模块 ReLU_5_a0 仅存在于第一个列表中。
模块 ReLU_16_15 仅存在于第二个列表中。
模块 ReLU_12_f2 仅存在于第一个列表中。
模块 ReLU_3_cd 仅存在于第一个列表中。
模块 ReLU_13_25 仅存在于第二个列表中。
模块 ReLU_3_2b 仅存在于第二个列表中。
模块 ReLU_11_9f 仅存在于第一个列表中。
模块 ReLU_12_39 仅存在于第一个列表中。
模块 ReLU_6_70 仅存在于第一个列表中。
模块 ReLU_9_20 仅存在于第二个列表中。
模块 ReLU_11_68 仅存在于第二个列表中。
模块 ReLU_7_af 仅存在于第一个列表中。
模块 ReLU_1_92 仅存在于第二个列表中。
模块 ReLU_2

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_9c
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_d9
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_bd
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_64
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_48
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_ea
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_59
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_5e
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_c9
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_c9
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_cb
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_09
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_5f
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_5c
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_15
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_28
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_28
Duplicate laye

模块 ReLU_4_59 仅存在于第二个列表中。
模块 ReLU_5_67 仅存在于第一个列表中。
模块 ReLU_12_66 仅存在于第二个列表中。
模块 ReLU_6_cb 仅存在于第二个列表中。
模块 ReLU_6_09 仅存在于第二个列表中。
模块 ReLU_3_ea 仅存在于第二个列表中。
模块 ReLU_16_e1 仅存在于第一个列表中。
模块 ReLU_14_d9 仅存在于第二个列表中。
模块 ReLU_10_f5 仅存在于第一个列表中。
模块 ReLU_10_b4 仅存在于第一个列表中。
模块 ReLU_13_27 仅存在于第一个列表中。
模块 ReLU_12_b8 仅存在于第二个列表中。
模块 ReLU_15_fa 仅存在于第二个列表中。
模块 ReLU_1_9c 仅存在于第二个列表中。
模块 ReLU_7_01 仅存在于第一个列表中。
模块 ReLU_11_cf 仅存在于第一个列表中。
模块 ReLU_8_28 仅存在于第二个列表中。
模块 ReLU_13_07 仅存在于第二个列表中。
模块 ReLU_16_ed 仅存在于第二个列表中。
模块 ReLU_14_9e 仅存在于第一个列表中。
模块 ReLU_8_15 仅存在于第二个列表中。
模块 ReLU_16_02 仅存在于第一个列表中。
模块 ReLU_9_4d 仅存在于第一个列表中。
模块 ReLU_3_cf 仅存在于第一个列表中。
模块 ReLU_4_05 仅存在于第一个列表中。
模块 ReLU_1_8d 仅存在于第一个列表中。
模块 ReLU_14_7c 仅存在于第一个列表中。
模块 ReLU_10_0b 仅存在于第二个列表中。
模块 ReLU_3_ee 仅存在于第一个列表中。
模块 ReLU_1_3d 仅存在于第一个列表中。
模块 ReLU_13_12 仅存在于第二个列表中。
模块 ReLU_16_15 仅存在于第二个列表中。
模块 ReLU_12_15 仅存在于第一个列表中。
模块 ReLU_15_8c 仅存在于第一个列表中。
模块 ReLU_7_ce 仅存在于第一个列表中。
模块 ReLU_9_0f 仅存在于第一个列表中。
模块 ReLU_8_a2 仅存在于第一个列表中。
模块 ReLU_4_5e 仅存在于第二个列表中。
模块 ReLU_2_bd 仅存在于第二个列表中。
模块 ReL

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_e4
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_50
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_cd
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_44
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_45
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_27
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_12
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_01
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_4a
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_97
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_62
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_93
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_8f
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_dd
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_e4
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_8a
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_cc
Duplicate laye

...

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_3e
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_30
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_9a
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_e4
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_3a
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_8a
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_e2
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_17
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_42
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_b8
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_89
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_2b
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_b1
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_51
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_4c
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_33
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_80
Duplicate laye

模块 ReLU_13_d6 仅存在于第二个列表中。
模块 ReLU_3_3a 仅存在于第二个列表中。
模块 ReLU_12_ba 仅存在于第一个列表中。
模块 ReLU_10_a1 仅存在于第一个列表中。
模块 ReLU_5_b8 仅存在于第二个列表中。
模块 ReLU_15_a6 仅存在于第二个列表中。
模块 ReLU_12_50 仅存在于第二个列表中。
模块 ReLU_11_b8 仅存在于第一个列表中。
模块 ReLU_7_bc 仅存在于第一个列表中。
模块 ReLU_3_5e 仅存在于第一个列表中。
模块 ReLU_4_e8 仅存在于第一个列表中。
模块 ReLU_10_f5 仅存在于第二个列表中。
模块 ReLU_1_3e 仅存在于第二个列表中。
模块 ReLU_1_d2 仅存在于第一个列表中。
模块 ReLU_14_7e 仅存在于第二个列表中。
模块 ReLU_11_d6 仅存在于第一个列表中。
模块 ReLU_4_e2 仅存在于第二个列表中。
模块 ReLU_4_17 仅存在于第二个列表中。
模块 ReLU_8_29 仅存在于第一个列表中。
模块 ReLU_6_2b 仅存在于第二个列表中。
模块 ReLU_7_de 仅存在于第一个列表中。
模块 ReLU_1_30 仅存在于第二个列表中。
模块 ReLU_6_e6 仅存在于第一个列表中。
模块 ReLU_7_b1 仅存在于第二个列表中。
模块 ReLU_14_04 仅存在于第二个列表中。
模块 ReLU_7_51 仅存在于第二个列表中。
模块 ReLU_1_06 仅存在于第一个列表中。
模块 ReLU_11_f8 仅存在于第二个列表中。
模块 ReLU_5_90 仅存在于第一个列表中。
模块 ReLU_4_a0 仅存在于第一个列表中。
模块 ReLU_13_9b 仅存在于第一个列表中。
模块 ReLU_9_26 仅存在于第一个列表中。
模块 ReLU_12_6e 仅存在于第一个列表中。
模块 ReLU_15_44 仅存在于第一个列表中。
模块 ReLU_2_9d 仅存在于第一个列表中。
模块 ReLU_9_80 仅存在于第二个列表中。
模块 ReLU_14_48 仅存在于第一个列表中。
模块 ReLU_3_8a 仅存在于第二个列表中。
模块 ReLU_16_cd 仅存在于第二个列表中。
模块 ReLU_2

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_f5
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_42
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_bc
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_d3
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_6c
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_3e
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_ac
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_0a
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_d8
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_5b
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_d5
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_59
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_c8
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_44
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_07
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_18
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_27
Duplicate laye

模块 ReLU_16_8d 仅存在于第二个列表中。
模块 ReLU_1_c1 仅存在于第一个列表中。
模块 ReLU_9_9e 仅存在于第一个列表中。
模块 ReLU_13_17 仅存在于第一个列表中。
模块 ReLU_16_98 仅存在于第二个列表中。
模块 ReLU_8_18 仅存在于第二个列表中。
模块 ReLU_7_44 仅存在于第二个列表中。
模块 ReLU_10_81 仅存在于第一个列表中。
模块 ReLU_13_0d 仅存在于第二个列表中。
模块 ReLU_6_44 仅存在于第一个列表中。
模块 ReLU_6_fe 仅存在于第一个列表中。
模块 ReLU_14_b3 仅存在于第一个列表中。
模块 ReLU_5_5b 仅存在于第二个列表中。
模块 ReLU_13_fe 仅存在于第二个列表中。
模块 ReLU_3_3e 仅存在于第二个列表中。
模块 ReLU_4_0a 仅存在于第二个列表中。
模块 ReLU_7_72 仅存在于第一个列表中。
模块 ReLU_15_1a 仅存在于第二个列表中。
模块 ReLU_12_fb 仅存在于第一个列表中。
模块 ReLU_11_cf 仅存在于第一个列表中。
模块 ReLU_14_56 仅存在于第二个列表中。
模块 ReLU_5_d8 仅存在于第二个列表中。
模块 ReLU_10_7e 仅存在于第二个列表中。
模块 ReLU_2_fa 仅存在于第一个列表中。
模块 ReLU_1_42 仅存在于第二个列表中。
模块 ReLU_2_bc 仅存在于第二个列表中。
模块 ReLU_12_c3 仅存在于第二个列表中。
模块 ReLU_4_3c 仅存在于第一个列表中。
模块 ReLU_5_65 仅存在于第一个列表中。
模块 ReLU_2_d3 仅存在于第二个列表中。
模块 ReLU_10_7c 仅存在于第一个列表中。
模块 ReLU_1_99 仅存在于第一个列表中。
模块 ReLU_9_6b 仅存在于第一个列表中。
模块 ReLU_8_07 仅存在于第二个列表中。
模块 ReLU_6_d5 仅存在于第二个列表中。
模块 ReLU_15_af 仅存在于第一个列表中。
模块 ReLU_14_28 仅存在于第二个列表中。
模块 ReLU_15_2c 仅存在于第一个列表中。
模块 ReLU_12_ec 仅存在于第一个列表中。
模块 ReLU

OOM when allocating 1004535808 bytes on device index=0 id='36243bf0f29a4daea824956e0a3c4105'
OOM when allocating 1004535808 bytes on device index=0 id='36243bf0f29a4daea824956e0a3c4105'


In [9]:
import pandas as pd
df = pd.json_normalize(result_list)

In [10]:
df["Estimate Diff"] = df["compare.est_diff"].apply(format_memory)
df

,model,batch,torch.est,torch.ground,torch.error,torch.diff,huggingface.est,huggingface.ground,huggingface.error,huggingface.diff,compare.forward,compare.backward,compare.est_diff,Estimate Diff
0,ConvNeXtTiny,10,369098752,312475648,18.12,56623104,354418688,262144000,35.20,92274688,True,True,-14680064,-14.00 MB
1,ConvNeXtTiny,50,994050048,998244352,0.42,-4194304,975175680,958398464,1.75,16777216,True,True,-18874368,-18.00 MB
2,ConvNeXtTiny,90,1614807040,1233125376,30.95,381681664,1604321280,1629487104,1.54,-25165824,True,True,-10485760,-10.00 MB
3,ConvNeXtTiny,130,2210398208,1625292800,36.00,585105408,2206203904,2281701376,3.31,-75497472,True,True,-4194304,-4.00 MB
4,ConvNeXtTiny,170,2824863744,1644167168,71.81,1180696576,2824863744,2925527040,3.44,-100663296,True,True,0,0 b
5,ConvNeXtTiny,210,3468689408,1667235840,108.05,1801453568,3468689408,3590324224,3.39,-121634816,True,True,0,0 b
6,ConvNeXtTiny,250,4106223616,4678746112,12.24,-572522496,4112515072,4280287232,3.92,-167772160,True,True,6291456,6.00 MB
7,ConvNeXtTiny,290,4720689152,1755316224,168.94,2965372928,4731174912,4945084416,4.33,-213909504,True,True,10485760,10.00 MB
8,ConvNeXtTiny,330,5368709120,3690987520,45.45,1677721600,5381292032,5599395840,3.90,-218103808,True,True,12582912,12.00 MB
9,ConvNeXtTiny,370,5911871488,5771362304,2.43,140509184,5928648704,6199181312,4.36,-270532608,True,True,16777216,16.00 MB



# Analysis of Shopshot Data between PyTorch and HuggingFace


In [8]:
model_name = model
batch = batch_size
torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
all_torch_dirs = os.listdir(torch_data_dir)
all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False and str(d).startswith("@") is False]
dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)

In [9]:
torch_snapshot_file = Path(filter_files(".pickle", dirs_sorted[0], fuzz=True)[0])
torch_profiler_file = Path(filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1])


In [10]:
# Get HuggingFace Data
huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
huggingface_snapshot_file_xmem = Path(filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1])

torch_snapshot_file

PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-VGG16-SGD-170-1/20241126-015141-d180/results/snapshot/snapshot_result-1732585963.pickle')

In [14]:
torch_snapshot = SnapshotAnalyser(torch_snapshot_file)
hugging_snapshot = SnapshotAnalyser(huggingface_snapshot_file_xmem)

In [15]:
torch_profiler = ProfilerDataProcessing(torch_profiler_file)

In [16]:
huggingface_profiler = ProfilerDataProcessing(huggingface_profiler_file_llm)
huggingface_profiler_file_llm

'/Users/jiaboshi/Documents/101-Data/002-xMem-LLM/002-HuggingFace/VGG16-170-LLM/results/callback/Profiler/Glaswigian-Researcher_123964.1739486916203614949.pt.trace.json'

In [17]:
layer_in_torch = torch_profiler.get_iteration(2).layer_summary()
layer_in_hugging = huggingface_profiler.get_iteration(2).layer_summary()

In [18]:
_, torch_estimator = get_xmem_memory(torch_profiler_file, batch_size, max_gpu_memory)
_, hugging_estimator = get_xmem_memory(huggingface_profiler_file_llm, batch_size, max_gpu_memory, huggingface_enabled=True)

In [19]:
training_memorys = torch_estimator.training_memory(iteration_index=2)
model_memory = torch_estimator.model_memory(iteration_index=2)
data_memory = torch_estimator.data_memory(iteration_index=2)
estimated_instance, estimated_result = torch_estimator.estimate_memory_blocks(training_memorys)
estimated_model_instance, estimated_model_result = torch_estimator.estimate_memory_blocks(model_memory)
estimated_data_instance, estimated_data_result = torch_estimator.estimate_memory_blocks(data_memory)

print(f"Model Memory: {format_memory(estimated_model_result['memory']['segment'])}\n"
      f"Training Memory: {format_memory(estimated_result['memory']['segment'])}\n"
      f"Data Memory: {format_memory(estimated_data_result['memory']['segment'])}")


Model Memory: 534.00 MB
Training Memory: 3.28 GB
Data Memory: 18.00 MB


In [20]:
hugging_memory = hugging_estimator.training_memory(iteration_index=2, zero_grad=False)
hugging_model_memory = hugging_estimator.model_memory(iteration_index=2)
hugging_data_memory = hugging_estimator.data_memory(iteration_index=2)
hugging_estimated_instance, hugging_estiamted_result = hugging_estimator.estimate_memory_blocks(hugging_memory)
hugging_estimated_memory_instance, hugging_model_memory_result = hugging_estimator.estimate_memory_blocks(hugging_model_memory)
hugging_estimatod_data_instance, hugging_data_memory_result = hugging_estimator.estimate_memory_blocks(hugging_data_memory)

print(f"Model Memory: {format_memory(hugging_model_memory_result['memory']['segment'])}\n"
      f"Training Memory: {format_memory(hugging_estiamted_result['memory']['segment'])}\n"
      f"HuggingFace Data Memory: {format_memory(hugging_data_memory_result['memory']['segment'])}")

Model Memory: 534.00 MB
Training Memory: 3.28 GB
HuggingFace Data Memory: 18.00 MB


In [15]:
profile_file = "/home/glaswigian/.cache/XMemEstimator/158ec996a9a344d881fa5f551e8bd58a/results/callback/Profiler/4f5fcf65edb9_1.1740663353153780165.pt.trace.json"
get_xmem_memory(
    profiler_file=profile_file,
    batch=50,
    max_in_gb=8,
    huggingface_enabled=True
)

ValueError: Time 1740663350947980, the amount of memory freed is not the same as the current block.Original Size: 819200 != free Size: -147456